# Gymnasium: BipedalWalker-v3

Our objective is to train an agent to navigate the BipedalWalker environment using Reinforcement Learning. Before implementing complex algorithms or aiming for advanced maneuvers (like doing a flip), we need to understand the environment's dynamics.

## Environment Overview
`BipedalWalker-v3` is a 2D physics simulation environment from the Gymnasium Box2D environments. The goal is to make a bipedal robot walk to the right end of the terrain. In the `hardcore=True` version, the terrain is not flat; it includes obstacles such as ladders, stumps, and pitfalls.

### Action Space
The action space is a continuous `Box(-1.0, 1.0, (4,), float32)`. The agent controls the robot by applying torques to its four main joints. The four values in the action array represent:
1. Hip 1 (Torque / Speed)
2. Knee 1 (Torque / Speed)
3. Hip 2 (Torque / Speed)
4. Knee 2 (Torque / Speed)

All action values must be within the `[-1.0, 1.0]` range.

### Reward System
The agent receives rewards based on its forward progress and energy efficiency. According to the official documentation, the reward is calculated as follows:
* **Forward Movement:** The agent is rewarded for moving forward (to the right). Reaching the end of the terrain yields a total of over 300 points.
* **Falling Penalty:** If the robot's main body (hull) touches the ground, it falls. This results in a heavy penalty of **-100** points, and the episode terminates immediately.
* **Motor Torque Penalty:** To encourage efficient, natural walking rather than chaotic flailing, applying motor torque costs a small negative reward.
### Set up the Environment

In [29]:
import gymnasium as gym
env = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human") # Human we can see the environment

## Baseline: Random Actions

To establish a baseline and visualize how an untrained agent interacts with the physics engine, we will run a single episode using a random policy. The agent will sample actions uniformly from the action space until the episode ends.

An episode ends if:
- `terminated` is True (the agent falls or reaches the goal).
- `truncated` is True (the agent runs out of time/steps).
- Past 20 seconds of simulation time.

In [33]:
import time

env = gym.make("BipedalWalker-v3", hardcore=False, render_mode="human") 
limit_time = 10  # seconds
start_time = time.time()
obs, info = env.reset()

terminated = False
truncated = False
total_reward = 0.0
step_count = 0

# Loop until the agent finishes or fails
while not (terminated or truncated) and (time.time() - start_time < limit_time):
    # Sample a random continuous action within [-1, 1] for the 4 joints
    action = env.action_space.sample() 
    
    # Step the environment forward
    obs, reward, terminated, truncated, info = env.step(action)
    
    total_reward += reward
    step_count += 1

# Close the rendering window
env.close()

print(f"Episode finished after {step_count} steps.")
print(f"Total Reward with random policy: {total_reward:.2f}")

Episode finished after 470 steps.
Total Reward with random policy: -27.31


### Changing the Environment
IInstead of making the agent learn to walk, we can make it learn to do a `flip`. To do that, we need to change the reward system and the enviroment itself, so that can be able to do a flip.

In [21]:
# code for sac